# Rebuilding the data pipeline from zero: a question-led guide

*Personal guide by Deric. Use it to rebuild this project (or a similar one) without
looking at the code first, and to train design and planning skills.*

**How to use it**
1. Read the questions of a step and write your own answer (a few lines) in a new cell.
2. Only then open **Guidance**: it says what we did and why.
3. Answer the **Reflect** questions: what could be better, what is missing, what can break.
4. At the end, compare your answers with `docs/decisions.md` and the journal
   (`deric_data_pipeline_journal.ipynb`).

**The habit to train:** for every choice, be able to say *what problem it solves*,
*what it costs*, and *what would prove it wrong*.

## Step 0: Framing the project

**Answer before reading the guidance**

1. What is the research question, in one sentence? Which hypotheses (H1, H2, H3) does the data have to serve?
2. For each hypothesis, which variables are needed, at which frequency, and over which period?
3. Which analysis needs the *date an event happened* and which needs the *date it became known*?
4. What would make the whole project worthless even if the code runs? (Hint: think about time.)
5. How will the work be split, reviewed and graded? What does that imply for the Git history?

<details>
<summary><b>Guidance (what we did and why)</b></summary>

- Question: the August 2024 yen carry unwind. H1 negative skew of carry returns; H2 extreme net-short yen positioning precedes lower forward carry returns; H3 high carry-to-volatility precedes worse tails.
- Needed: USD/JPY and other assets (daily), US and JP short rates (carry), CFTC positioning (weekly), BIS yen credit (quarterly), MoF interventions (events).
- Event study → event date; predictive tests → publication date. Mixing them is the biggest risk.
- Look-ahead bias: using a value before it was published makes predictive results look better than reality.
- Graded on history ⇒ one branch per topic, PRs, a reviewer who did not write the code, branches kept.

</details>

**Reflect**

- Which hypothesis is the weakest given the data available? Would you change it?
- Is the August 2024 episode one event or several (BoJ hike 31 Jul, US payrolls 2 Aug, 5 Aug crash)? How would that change the design?
- What data would an industry desk use that we do not have (e.g. options skew, OIS rates, intraday FX)?

## Step 1: Repository and environment

**Answer before reading the guidance**

1. What folder structure lets five people work without stepping on each other?
2. Where do settings (dates, URLs, tickers, lags) live, and why only there?
3. How does `from src import config` work both from the repo root and from a notebook in `notebooks/`?
4. Should data files be in Git? What replaces them?
5. What protects `main` from mistakes?

<details>
<summary><b>Guidance (what we did and why)</b></summary>

- `src/data/` one module per source, `src/cleaning.py`, analysis modules, `tests/`, `notebooks/` for exploration, `docs/decisions.md`, `main.ipynb` final.
- `src/config.py`: one place to change a date or a lag; every estimated constant has a comment saying how it was checked.
- `pyproject.toml` + `pip install -e .` (Adam's fix). Without the install, only `python -m src...` works: the 'No module named src' error.
- `data/` is ignored; anyone rebuilds it with the fetch commands. Downside: sources revise, so two people may get different numbers.
- Branch protection on `main` (PR required, no force-push), reviews, tests run before merging.

</details>

**Reflect**

- Our `requirements.txt` is not pinned to exact versions. What could break in six months (yfinance, pandas)?
- Tests only run on a laptop. What would continuous integration (GitHub Actions) catch, and what would it need (data access, time)?
- How would you snapshot the data used for the final results so the grader gets the same numbers?

## Step 2: Retrieving each source

**Answer before reading the guidance**

1. For each source: what is the *stable key* that identifies the series (ticker, series id, contract code)?
2. What are the units and scale? Which encoding? Which date convention?
3. Which errors would be *silent* (no crash, wrong numbers)? List three.
4. Should `fetch_*` save the file itself? Why or why not?
5. What should happen if the source changes its format tomorrow?

<details>
<summary><b>Guidance (what we did and why)</b></summary>

- Keys: Yahoo tickers, FRED series ids (`IR3TIB01USM156N`, `IR3TIB01JPM156N`), CFTC code `097741` (read as text: leading zero), BIS dataflow key, MoF CSV columns.
- Units: BIS millions of JPY (`UNIT_MULT = 6`) → trillions; MoF 100 million yen → billions, signed; FRED percent. MoF file is Shift-JIS (`cp932`).
- Silent errors met: FRED columns renamed by position (swap US/JP); yfinance excludes the `end` day; CFTC market name changed over time; Yahoo one-day bad prints.
- Return the DataFrame, save in a separate function or the `__main__` block: tests and notebooks can call it without writing files.
- Validate on download and raise `ValueError` with a clear message (fail loudly).

</details>

**Reflect**

- We use the CFTC *Legacy* report (non-commercials). Would the *Traders in Financial Futures* report (leveraged funds, asset managers) measure carry traders better?
- Yahoo FX closes: at what time is the USD/JPY close taken? How would you find out, and does it matter for the event study?
- Gold and Brent are continuous futures. How would you detect and handle roll jumps?
- Which source is most likely to disappear or change format first? How would you notice?

## Step 3: Validation (integrity, content, timing)

**Answer before reading the guidance**

1. What does 'validated' mean for a dataset? Give three different kinds of checks.
2. Which checks belong in the download code, and which in `tests/`?
3. Why is checking the *release date* part of validation, not just cleaning?
4. For each release rule (FRED +3 months, BIS +4 months, MoF quarter end +45 days, CFTC next Friday), what evidence would prove it wrong?
5. You see a strange jump in a series. What is your procedure?

<details>
<summary><b>Guidance (what we did and why)</b></summary>

- Integrity (is the file what I think: units, duplicates, columns), content (plausible and consistent values: totals, known historical values, ranges), timing (when was each value public).
- Download code: conditions that make the data unusable → raise. Tests: properties of the saved file → assert; skip if the file is missing.
- Correct data used too early invalidates predictive results. The release rule is a claim; it needs evidence like any number.
- Evidence used: FRED page 'Updated 15 Oct 2026' (US to Sep, JP to Aug); BIS 2026-Q2 out 29 Oct; MoF quarterly release dates; CFTC shutdown notices (back on schedule 8 Nov 2013, 8 Mar 2019).
- Procedure (JP rate 2025-01): measure the anomaly (38bp vs a 25bp hike) → list hypotheses (rounding vs new source) → cheapest separating test (JBA TIBOR PDFs) → compare *exact* values (month-end matched exactly; averages only nearly) → decide and log.

</details>

**Reflect**

- Past release dates were not checked (ALFRED needs an API key). How big could the error be, and in which direction?
- The MoF check allows a gap of 5 (×100m yen) per quarter. Is that tolerance justified, or just convenient?
- Which of our tests would still pass if the data were wrong? (A test that cannot fail proves nothing.)
- What independent source could cross-check CFTC or BIS like JBA TIBOR did for FRED?

## Step 4: Cleaning rules

**Answer before reading the guidance**

1. What should the index of each table be: the event date or the publication date? Test your answer on MoF.
2. Should missing prices be filled? What does filling do to returns?
3. When is removing an outlier legitimate? How do you avoid removing real crashes?
4. Why fix the end date of the sample instead of using today?
5. Where do you record each rule so the team can review it?

<details>
<summary><b>Guidance (what we did and why)</b></summary>

- Index = the date the value refers to; `release_date` as a column. Indexing MoF by release date puts all days of a quarter on one date (duplicates).
- No: filling creates fake zero returns. Keep NaN; align in the merge step.
- Only for *data errors*: USD/JPY one-day spikes >3% that reverse while neighbours agree (e.g. 109.24 between 92.82 and 92.30). Not for VIX, Bitcoin, oil, USD/MXN where reversing jumps are real.
- `date.today()` changes results on every run; `END = 2026-09-30` makes them reproducible.
- `docs/decisions.md`: date, source, decision, reason, author; mark proposals as pending until approved; strike through superseded rules.

</details>

**Reflect**

- We approved PR #20 (BIS by release date) and had to undo it. What question, asked at review time, would have caught it?
- The 2025 JP break is kept and flagged. When would replacing the series (JBA TIBOR for the whole period) be the better choice?
- Which cleaning rule are you least sure about? What result would change if it were wrong?

## Step 5: Merging into one table

**Answer before reading the guidance**

1. Which calendar should the merged table use, and why?
2. For a monthly rate, what is the difference between 'the rate of this month' and 'the rate known today'? Which analysis needs which?
3. How do you make a value usable only after its release? On the release day, or the day after?
4. When is forward-filling correct, and when does it invent data?
5. What should happen if an intervention day has no USD/JPY price?
6. How do you test 'no look-ahead' directly?

<details>
<summary><b>Guidance (what we did and why)</b></summary>

- USD/JPY trading days: the carry trade is priced there. Other markets' holidays stay NaN; Bitcoin weekends are dropped.
- Rate of the month (`us_3m`, `rate_diff`) → carry accrual and event study. Known rate (`*_known`) → signals for H2/H3.
- `as_known`: index by `release_date`, shift +1 day, `reindex(calendar, method='ffill')`. The day after, because release dates are estimates and data comes out during the day (CFTC at 15:30 New York).
- Correct for 'latest known value' (`ffill` of releases, including over a NaN release such as FRED us_3m 2020-04). Wrong for 'value of this period': `rates_as_of` matches on the month so an unpublished month stays NaN.
- Stop with an error (never drop an intervention silently); a test checks that totals match the MoF file.
- For each day d, the `_known` value equals the latest release strictly before d. Test it on dense samples: a sparse test (every 250th day) hid a 22-day gap.

</details>

**Reflect**

- Adam's model uses weekly Fridays. With our rule, a Friday decision uses last week's CFTC report. Is a 10-day-old signal still informative? How would you test it?
- Bitcoin's Friday→Monday change includes the weekend. Does that bias volatility or tail estimates?
- MoF days after the last published quarter show 0 instead of 'unknown'. When could that mislead the analysis?
- Should the merged table carry a `jp_rate_break` flag column instead of relying on a config constant?

## Step 6: Git workflow and collaboration

**Answer before reading the guidance**

1. What is your branch strategy for one topic? For a topic that depends on another unmerged topic?
2. Before `switch`, `pull` or `merge`, what do you always check?
3. You get a conflict. How do you decide which side to keep *without guessing*?
4. Which merge method (merge commit, squash, rebase) do you choose, and what does each do to commit ids?
5. A commit landed on `main` by mistake and someone reverted it. How do you get the work back?
6. When is rewriting pushed history acceptable? What protects teammates?

<details>
<summary><b>Guidance (what we did and why)</b></summary>

- One branch per topic from up-to-date `main`; stacked branch (`data-cleaning` on `final-validation`) only when it truly depends on it; merge the base into it when the base changes.
- `git status` clean; no editor tab with ● (unsaved copies survive Git changes and can overwrite files when saved).
- Compare content with `git diff --stat <commit> <commit>`: empty means identical. Prove which side already contains everything, resolve, run the tests.
- Merge commit keeps ids; squash makes one new commit; *rebase and merge* copies commits with new ids, so a branch stacked on the old ones conflicts later (our #25 `decisions.md` conflict).
- `git revert <revert-sha>` (revert the revert), through a branch and a PR because `main` is protected.
- Only when necessary and quickly: backup branch, `--force-with-lease=<branch>:<expected sha>`, tell everyone to `git reset --hard origin/<branch>`.

</details>

**Reflect**

- Which of our incidents were caused by process (no branch, unsaved tabs) rather than by Git itself? What single habit prevents each?
- Would a short CONTRIBUTING.md with these rules have avoided the accident on `main`?
- How would you present the Git history to a grader so the collaboration is visible?

## Step 7: Looking back: what is missing?

Answer each question in a few lines, then rank the answers by *impact on results* × *effort*.

**What did we not check?**
- Release dates before 2026 (no ALFRED vintages). Revisions: we use today's values for the past
  (*vintage problem*); FRED and BIS revise.
- The time of day of each Yahoo close, and whether USD/JPY and US markets are aligned for the event study.
- Data quality of the series other than USD/JPY (gold/Brent rolls, VIX, USD/MXN).

**What could be wrong in the design?**
- 3-month interbank rates as the carry proxy: carry traders fund overnight or 1-month; OIS
  or policy rates could differ, especially around the 2025 break.
- Positioning measured by CFTC non-commercials: only exchange futures, a small part of yen carry
  (most is in OTC forwards and loans; BIS credit is a partial proxy).
- One event (August 2024) for the event study: how general are the conclusions?

**What can break in practice?**
- A source changes its format or URL (MoF CSV, BIS API version, CFTC zip names).
- Library updates change behaviour (yfinance `end`, pandas `ffill` / frequency aliases like `"QE"`).
- Someone edits `config.py` (`END`, a lag) without re-running everything.

**What is the next step?**
- Modelling table: Friday rows, forward 1-week and 4-week carry returns, features from `_known` columns.
- Quantile regressions (tails at 5% and 10%), walk-forward validation (10-year first window),
  benchmarks (historical quantile, volatility-scaled normal), robustness without the JP break.
- Snapshot the final data, pin library versions, add CI for the tests.

## Step 8: Self-assessment

Score yourself 1–5 now, and again after the next project.

| Skill | Question to ask yourself | Score |
|-------|--------------------------|-------|
| Framing | Could I explain which date each analysis needs, and why, in two sentences? | |
| Data sense | Do I know the units, frequency and publication lag of every series by heart? | |
| Validation | For every important number, do I know what would prove it wrong? | |
| Design | Can I justify each column of the merged table and its intended use? | |
| Planning | Did I see the dependencies (cleaning needs validation) before starting? | |
| Git | Can I resolve a conflict by proving content, not by guessing? | |
| Communication | Did I explain changes to teammates *before* they were surprised by them? | |

**Final reflexive questions**
1. If you had one more week, what would you do first, and why that before anything else?
2. Which decision would you defend most strongly in the defense? Which one most weakly?
3. What did a teammate's review catch that you missed? What does that say about your blind spots?
4. If you restarted from zero tomorrow, what would you do in the first hour?